# 08 - MCP 服务器集成

> **何时使用**: 当你想让 AI 助手（Claude Desktop、Cursor）直接操作数据库生成测试数据时。
>
> **核心概念**: MCP (Model Context Protocol) 让 AI 助手通过 3 个工具直接调用 sqlseed。

## 适用场景

- 用自然语言让 AI 生成测试数据 → MCP Server
- AI 助手需要查看数据库 schema → `sqlseed_inspect_schema`
- AI 助手需要生成配置 → `sqlseed_generate_yaml`
- AI 助手需要执行填充 → `sqlseed_execute_fill`

## 你将学到

- MCP Server 安装与配置
- 3 个 MCP 工具的使用
- 安全验证机制
- 资源 URI 访问

**📚 教程导航**

| 序号 | 主题 | 架构层 | 前置要求 |
|------|------|--------|----------|
| 01 | 快速上手与核心流程 | Orchestrator | 无 |
| 02 | 9 级策略链详解 | Core: ColumnMapper | 01 |
| 03 | 生成器与 Provider 体系 | Generators | 01 |
| 04 | 数据库层与多表关联 | Database + Core | 01 |
| 05 | 表达式派生与约束求解 | Core: DAG / Expression | 01 |
| 06 | 配置驱动与 Transform | Config / Core | 01 |
| 07 | AI 智能配置 | Plugins: AI | 01 |
| **→ 08** | **MCP 服务器集成** | **Plugins: MCP** | **07** |
| 09 | 插件系统与 Hook 生命周期 | Plugins | 01 |
| 10 | CLI 参考手册 | CLI | 06 |
| 11 | 工具类参考 | Utils | 01 |
| 12 | 测试集成模式 | Testing | 01 |

---

In [1]:
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys; sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 架构定位

| 模块 | 文件 | 核心类/函数 |
|------|------|------------|
| MCP 资源与工具 | `plugins/mcp-server-sqlseed/src/mcp_server_sqlseed/server.py` | `mcp` |

> 对应架构图: [§10 MCP 服务器架构](../docs/architecture.zh-CN.md#10-mcp-服务器架构)

## 1. 先看效果 — AI 助手直接操作数据库

MCP (Model Context Protocol) 让 AI 助手（Claude Desktop、Cursor）直接调用 sqlseed — **无需手动写代码**：

```
你: "分析 app.db 的 projects 表结构，生成 YAML 配置，填充 5000 行数据"
AI: [自动调用 inspect_schema → generate_yaml → execute_fill]
```

下面演示 3 个 MCP 工具的本地调用。

## 2. 为什么需要 MCP Server？

MCP (Model Context Protocol) 让 AI 助手（Claude Desktop、Cursor 等）直接操作数据库：

- **inspect**: AI 查看 schema，理解表结构
- **generate_yaml**: AI 根据 schema 生成数据配置
- **execute_fill**: AI 执行填充，生成测试数据

无需手动写代码或 CLI 命令 — AI 助手一站式完成。

## 3. Claude Desktop / Cursor 配置

在 AI 助手的 MCP 配置中添加 sqlseed 服务器：

In [2]:
import json

config = {
    'mcpServers': {
        'sqlseed': {
            'command': 'python',
            'args': ['-m', 'mcp_server_sqlseed'],
            'env': {'OPENROUTER_API_KEY': 'your-key-here'}
        }
    }
}

print('Claude Desktop / Cursor MCP 配置:')
print(json.dumps(config, indent=2))

Claude Desktop / Cursor MCP 配置:
{
  "mcpServers": {
    "sqlseed": {
      "command": "python",
      "args": [
        "-m",
        "mcp_server_sqlseed"
      ],
      "env": {
        "OPENROUTER_API_KEY": "your-key-here"
      }
    }
  }
}


## 4. 工具 1: sqlseed_inspect_schema

AI 助手调用此工具查看数据库 schema，返回列信息、FK、索引和样本数据。

In [3]:
import json

from mcp_server_sqlseed.server import sqlseed_inspect_schema

result = sqlseed_inspect_schema(str(db_path), table_name='organizations')
print('sqlseed_inspect_schema 结果 (dict):')
print(json.dumps(result, indent=2, ensure_ascii=False)[:500])

sqlseed_inspect_schema 结果 (dict):
{
  "organizations": {
    "table_name": "organizations",
    "columns": [
      {
        "name": "org_code",
        "type": "VARCHAR(16)",
        "nullable": false,
        "default": null,
        "is_primary_key": true,
        "is_autoincrement": false
      },
      {
        "name": "name",
        "type": "VARCHAR(64)",
        "nullable": false,
        "default": null,
        "is_primary_key": false,
        "is_autoincrement": false
      },
      {
        "name": "parent_code",
 


## 5. 工具 2: sqlseed_generate_yaml

AI 分析 schema 并生成 YAML 配置。需要 API Key。

In [4]:
import os

api_key = os.environ.get('OPENROUTER_API_KEY') or os.environ.get('OPENAI_API_KEY')

if api_key:
    from mcp_server_sqlseed.server import sqlseed_generate_yaml
    result = sqlseed_generate_yaml(str(db_path), table_name='organizations', api_key=api_key)
    print('sqlseed_generate_yaml 结果:')
    print(result[:500] if isinstance(result, str) else str(result)[:500])
else:
    print('No API key. Example AI-generated YAML:')
    print('''tables:
  - name: organizations
    count: 100
    columns:
      - name: org_code
        generator: pattern
        params:
          pattern: "ORG-\\d{4}"
      - name: name
        generator: company
      - name: description
        generator: sentence
        params:
          nb_words: 8''')

sqlseed_generate_yaml 结果:
db_path: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db
provider: mimesis
locale: en_US
tables:
- name: organizations
  count: 1000
  columns:
  - name: org_code
    generator: pattern
    params:
      regex: ORG-[0-9]{4,6}
  - name: name
    generator: company
  - name: parent_code
    generator: foreign_key
    params:
      ref_table: organizations
      ref_column: org_code
  - name: description
    generator: text
    params:
      min_length: 50
      max_length: 200
  -


## 6. 工具 3: sqlseed_execute_fill

执行数据填充，支持内联 YAML 配置。

In [5]:
import json

from mcp_server_sqlseed.server import sqlseed_execute_fill

result = sqlseed_execute_fill(str(db_path), table_name='tags', count=3)
print('sqlseed_execute_fill 结果:')
print(json.dumps(result, indent=2, ensure_ascii=False)[:500])

Generating tags:   0%|          | 0/3 [00:00<?, ?it/s]

sqlseed_execute_fill 结果:
{
  "table_name": "tags",
  "count": 3,
  "elapsed": 0.024562137987231836,
  "errors": []
}


## 7. 安全验证

MCP Server 内置安全验证，防止路径遍历和 SQL 注入。

In [6]:
from mcp_server_sqlseed.server import _validate_db_path, _validate_table_name

print('安全验证函数:')
print()

# Valid path (existing .db file)
try:
    result = _validate_db_path(str(db_path))
    print(f'  _validate_db_path("{db_path.name}") -> OK (resolved: {result})')
except Exception as e:
    print(f'  _validate_db_path("{db_path.name}") -> {type(e).__name__}: {e}')

# Invalid paths
for path in ['../../../etc/passwd', 'test.txt', 'nonexistent.db']:
    try:
        _validate_db_path(path)
        print(f'  _validate_db_path("{path}") -> OK')
    except Exception as e:
        print(f'  _validate_db_path("{path}") -> {type(e).__name__}: {e}')

# Table name validation (requires allowed_tables list)
allowed = ['organizations', 'members', 'projects', 'tasks', 'tags']
for name in ['organizations', '; DROP TABLE --', '../hack']:
    try:
        _validate_table_name(name, allowed)
        print(f'  _validate_table_name("{name}") -> OK')
    except Exception as e:
        print(f'  _validate_table_name("{name}") -> {type(e).__name__}: {e}')

安全验证函数:

  _validate_db_path("sqlseed_demo.db") -> OK (resolved: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db)
  _validate_db_path("../../../etc/passwd") -> ValueError: Invalid database path: ../../../etc/passwd. Must be a .db, .sqlite, or .sqlite3 file.
  _validate_db_path("test.txt") -> ValueError: Invalid database path: test.txt. Must be a .db, .sqlite, or .sqlite3 file.
  _validate_db_path("nonexistent.db") -> ValueError: Database file not found: nonexistent.db
  _validate_table_name("organizations") -> OK
  _validate_table_name("; DROP TABLE --") -> ValueError: Table '; DROP TABLE --' does not exist in the database. Available: ['organizations', 'members', 'projects', 'tasks', 'tags']
  _validate_table_name("../hack") -> ValueError: Table '../hack' does not exist in the database. Available: ['organizations', 'members', 'projects', 'tasks', 'tags']


## 总结

| MCP 工具 | 功能 | 需要 API Key |
|----------|------|:------------:|
| `sqlseed_inspect_schema` | 查看 schema | ❌ |
| `sqlseed_generate_yaml` | AI 生成配置 | ✅ |
| `sqlseed_execute_fill` | 执行填充 | ❌ |

**下一步**: [09-plugin-hooks.ipynb](09-plugin-hooks.ipynb) — 插件系统与 Hook 生命周期

In [7]:
# ✅ 验证: 确保数据已成功生成并写入
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # 基本行数验证
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
